# AI Job Board — Prototype Notebook (Colab)

This notebook walks through the **data + AI pipeline** behind the Job Board without needing to stand up the Flask server — useful for demoing/explaining the approach.

No API key is used anywhere in this notebook.

**Steps:** 1) load the multi-platform dataset  2) AI tagging  3) dedup  4) resume-based recommendations  5) the conversational assistant logic.

> To run the actual web app locally instead, use the downloaded `jobboard/` folder with `python app.py` (see README.md).

## 0. Setup
If you're running this in Colab standalone (not next to the `jobboard/` folder), the next cell installs dependencies and recreates the minimal set of modules needed. If you uploaded the whole `jobboard/` folder to Colab (e.g. via the Files panel or `git clone`), just `%cd jobboard` instead and skip straight to Step 1.

In [ ]:
!pip install -q scikit-learn pypdf python-docx

# If you uploaded the jobboard/ project folder to Colab, uncomment:
# %cd jobboard
# import sys; sys.path.insert(0, '.')

# Otherwise this cell will look for utils/tagger.py etc. next to this
# notebook. If you're running the notebook standalone, upload the
# `utils/`, `data/`, and `database.py` files alongside it in the Colab
# file browser first.

## 1. Load the multi-platform dataset
This loads `data/jobs_raw.json`. **Replace this file with the real dataset** downloaded from the Google Drive link in the assignment brief — the schema (source/title/company/description/...) is the same.

In [ ]:
import json

with open('data/jobs_raw.json') as f:
    raw_jobs = json.load(f)

print(f'Loaded {len(raw_jobs)} raw job records')
raw_jobs[0]

In [ ]:
import pandas as pd
df = pd.DataFrame(raw_jobs)
df['source'].value_counts()

## 2. AI-based tagging (skills, role category, experience)
Uses `utils/tagger.py` — a curated skill dictionary matched with word-boundary-safe regex, plus keyword-based role classification and regex-based experience extraction. No external API required.

In [ ]:
from utils.tagger import enrich_job

enriched_jobs = [enrich_job(dict(j)) for j in raw_jobs]
enriched_jobs[0]

In [ ]:
from collections import Counter
all_skills = Counter(s for j in enriched_jobs for s in j['skills'])
all_skills.most_common(15)

## 3. Deduplication
A content hash of normalized (title, company, location) collapses cross-platform duplicates while keeping track of every source a job was posted on. See `database.py` docstring for the full rationale.

In [ ]:
import os
if os.path.exists('jobboard.db'):
    os.remove('jobboard.db')

from database import init_db, upsert_job, get_all_jobs
init_db()

stats = {'inserted': 0, 'duplicate': 0, 'skipped': 0}
for j in enriched_jobs:
    stats[upsert_job(j)] += 1

print(stats)
print(f"{len(get_all_jobs())} unique jobs stored after dedup")

## 4. Resume-based recommendations
Paste in (or upload) resume text, extract skills with the same tagger used for jobs, then rank all jobs by `0.6 * TF-IDF cosine similarity + 0.4 * skill overlap`.

In [ ]:
sample_resume_text = '''
John Doe — Data Analyst
2 years experience with Python, SQL, Power BI, Excel, Statistics, Tableau, Machine Learning.
Built dashboards and automated reporting pipelines using Pandas.
'''

from utils.tagger import extract_skills
from utils.recommender import recommend_jobs

resume_skills = extract_skills(sample_resume_text)
print('Extracted resume skills:', resume_skills)

all_jobs = get_all_jobs()
recs = recommend_jobs(sample_resume_text, resume_skills, all_jobs, top_n=5)

for r in recs:
    print(f"{r['match_score']:>5.1f}%  {r['title']} @ {r['company']} ({r['source']})  shared: {r['matched_skills']}")

## 5. Conversational AI Job Assistant
Intent-detection + grounded-template answers (fully offline). Try changing the questions below — supports suitability, missing skills, explaining a JD, which jobs to apply to, prep guidance, comparing two jobs, and resume improvement.

In [ ]:
from utils.assistant import answer

target_job = all_jobs[0]
compare_job_example = all_jobs[1]

print('JOB:', target_job['title'], '@', target_job['company'])
print()
print('Q: Am I suitable for this job?')
print('A:', answer('Am I suitable for this job?', resume_skills=resume_skills, job=target_job))
print()
print('Q: What skills am I missing?')
print('A:', answer('What skills am I missing?', resume_skills=resume_skills, job=target_job))
print()
print('Q: Explain this job description.')
print('A:', answer('Explain this job description.', resume_skills=resume_skills, job=target_job))
print()
print('Q: Compare these jobs.')
print('A:', answer('Compare these jobs.', resume_skills=resume_skills, job=target_job, compare_job=compare_job_example))

## (Optional) Run the full Flask app inside Colab
You can expose the Flask app publicly from Colab using `pyngrok`, though for the actual submission, running it locally via Docker/`python app.py` (see README) is more reliable/reproducible.

In [ ]:
# !pip install -q flask pyngrok
# from pyngrok import ngrok
# public_url = ngrok.connect(5000)
# print(public_url)
# !python app.py